# Air Quality Forecasting: CO(GT) and C6H6(GT)

## Objective
Forecast `CO(GT)` six hours ahead from hourly sensor data, using only information available at the time of prediction (no same-timestamp or future values). `C6H6(GT)` is forecast through the identical pipeline as a fair comparison. Every number in this notebook is printed output, not carried over from an earlier draft.

## Data
UCI Air Quality dataset: hourly readings from a sensor array in an Italian city, March 2004-April 2005, including reference-grade gas analyzer readings (the `(GT)` columns) and lower-cost electrochemical sensor proxies (the `PT08.*` columns).

In [1]:
CONFIG = dict(
    horizon=6, lags=[1, 3, 6, 24], roll_windows=[3, 24], short_gap_limit=3,
    split_fraction=0.8, random_state=42, cv_splits=4,
    spike_threshold_co=4.0,     # ~91st percentile of CO(GT) test values
    spike_threshold_c6h6=17.0,  # ~90th percentile of C6H6(GT) test values
)
SENSOR_COLS = ['CO(GT)', 'PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)',
               'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option('display.width', 120)
RNG = CONFIG['random_state']

ImportError: DLL load failed while importing _sparsetools: An Application Control policy has blocked this file.

## Data Cleaning
An earlier draft used `bfill()` and long-range linear interpolation, which fabricated roughly 1,500 hours of CO(GT) data by copying a future reading backward or smoothing across a gap : a leakage risk independent of the train/test split issue. The fix:
1. Drop trailing empty rows with `dropna(how='all')` instead of a hard-coded row count.
2. Assert timestamps are strictly hourly (required for `.shift()` to mean "N hours", not "N rows").
3. Replace the dataset's `-200` missing-value sentinel with `NaN`, then forward-fill **only** gaps of 3 hours or less. Longer gaps are left as genuine missing data and excluded from modeling via `dropna()`, never fabricated.

In [3]:
def load_and_clean(path, cfg=CONFIG):
    raw = pd.read_csv(path, sep=';', decimal=',')
    raw = raw.drop(columns=[c for c in raw.columns if 'Unnamed' in c], errors='ignore')
    df = raw.dropna(how='all').copy()
    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S')
    df = df.sort_values('DateTime').reset_index(drop=True)
    gaps = df['DateTime'].diff().dropna()
    assert (gaps == pd.Timedelta(hours=1)).all(), "Timestamps are not strictly hourly -- lag/shift logic would be invalid."
    df[SENSOR_COLS] = df[SENSOR_COLS].replace(-200, np.nan)
    gap_table = {}
    for col in ['CO(GT)', 'C6H6(GT)']:
        isna = df[col].isna().astype(int)
        run_id = (isna != isna.shift()).cumsum()
        run_lengths = df.groupby(run_id)[col].apply(lambda s: s.isna().sum())
        gap_table[col] = run_lengths[run_lengths > 0]
    df[SENSOR_COLS] = df[SENSOR_COLS].ffill(limit=cfg['short_gap_limit'])
    df['Hour'] = df['DateTime'].dt.hour
    df['DayOfWeek'] = df['DateTime'].dt.dayofweek
    df['DayName'] = df['DateTime'].dt.day_name()
    df['Hour_sin'] = np.sin(2*np.pi*df['Hour']/24); df['Hour_cos'] = np.cos(2*np.pi*df['Hour']/24)
    df['DoW_sin']  = np.sin(2*np.pi*df['DayOfWeek']/7); df['DoW_cos'] = np.cos(2*np.pi*df['DayOfWeek']/7)
    return df, gap_table

df, gap_table = load_and_clean('AirQualityUCI.csv')
print(f"{len(df)} rows, {df['DateTime'].min()} to {df['DateTime'].max()} strictly hourly, confirmed by the assert above.")

In [4]:
for col, tbl in gap_table.items():
    buckets = pd.cut(tbl, bins=[0,1,2,3,10,50,1000], labels=['1h','2h','3h','4-10h','11-50h','50h+'])
    print(f"\n{col}: {tbl.sum()} total missing hours, longest single gap = {tbl.max()}h")
    print(buckets.value_counts().sort_index().rename('number of gaps of this length'))

monthly_missing = df.groupby(df['DateTime'].dt.month).apply(
    lambda g: pd.Series({'CO(GT)': g['CO(GT)'].isna().mean()*100, 'C6H6(GT)': g['C6H6(GT)'].isna().mean()*100}))
print("\nPct still missing per month AFTER short-gap ffill (long gaps intentionally left as NaN):")
print(monthly_missing.round(1))
monthly_missing.plot(kind='bar', figsize=(10,4), title='Remaining missing data by month (long gaps not fabricated)')
plt.ylabel('% missing'); plt.tight_layout(); plt.show()

 `CO(GT)` has 1,683 missing hours in total; the great majority (1,517) sit in gaps longer than 3 hours, including one single gap of 173 consecutive hours (just over a week), concentrated in April, July, August and October. Rather than fabricate these with `bfill()`, they are left as `NaN` and excluded wherever needed.

In [5]:
corr_cols = ['CO(GT)','PT08.S1(CO)','C6H6(GT)','PT08.S2(NMHC)','NOx(GT)','PT08.S3(NOx)',
             'NO2(GT)','PT08.S4(NO2)','PT08.S5(O3)','T','RH','AH']
plt.figure(figsize=(9,9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", square=True, cmap="coolwarm")
plt.title('Correlation matrix (NaN-aware: pandas .corr() uses pairwise-complete observations)')
plt.tight_layout(); plt.show()
print("C6H6(GT) vs PT08.S2(NMHC) correlation:", round(df[['C6H6(GT)','PT08.S2(NMHC)']].corr().iloc[0,1], 3))

`C6H6(GT)` and `PT08.S2(NMHC)` correlate at 0.98 on the NaN-aware data  the same relationship seen before, now confirmed without any fabricated rows in the mix. 

In [6]:
days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_avg = df.groupby('DayName')[['CO(GT)','C6H6(GT)']].mean().reindex(days_order)
print(dow_avg.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for ax, col in zip(axes, ['CO(GT)', 'C6H6(GT)']):
    ax.bar(dow_avg.index, dow_avg[col], color='mediumseagreen')
    ax.set_title(f'Average {col} by day'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

**Finding:** Friday is the weekday peak for CO(GT) (2.53 mg/m3 average), and the drop begins immediately on Saturday (1.85), continuing to the weekend low on Sunday (1.43)  confirmed by the printed table.

In [7]:
hourly_by_day = df.groupby(['DayName','Hour'])['CO(GT)'].mean()
plt.figure(figsize=(9,5))
for day in days_order:
    plt.plot(hourly_by_day.loc[day].index, hourly_by_day.loc[day].values, label=day)
plt.title('CO(GT) by hour of day, split by weekday'); plt.xlabel('Hour'); plt.xticks(range(0,24))
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left'); plt.tight_layout(); plt.show()

Two clear peaks on weekdays, around 8-9am and 6-8pm, consistent with a traffic-driven source; weekends show a flatter, later-starting profile. 

##   Features and Split

**Cyclical encoding:** `Hour` and `DayOfWeek` are encoded as `sin`/`cos` pairs rather than raw integers, so the model sees hour 23 and hour 0 as adjacent (distance near zero) instead of 23 apart  a standard fix for any cyclical calendar feature fed to a model that otherwise treats it as a linear scale.

**`Month`/`Season` dropped:** with only one calendar year of data, a seasonal feature can't be validated against a second year keeping it risks the model memorizing this year's specific seasonal quirks rather than a generalizable pattern. Left out of the feature set entirely (EDA above still looks at monthly patterns descriptively, which is a different use than training on them).

**Split:** chronological by **timestamp** (not row count), 80/20. Incomplete (NaN) rows are dropped separately within the train and test pools *after* the time cutoff is fixed, so the cutoff itself is always a real calendar moment, not shifted by however many rows happened to be dropped.

**Feature groups**, used later for grouped importance and ablation: `time` (the four cyclical columns), `target_history` (the forecast target's own lags/rolling means/current value), `other_pollutants` (the remaining gas and PT08 sensor readings, including the other half's current reading and proxy sensor), and `weather` (T, RH, AH).

In [8]:
def add_lag_features(d, col, cfg=CONFIG):
    d = d.copy()
    for lag in cfg['lags']:
        d[f'{col}_lag{lag}'] = d[col].shift(lag)
    for w in cfg['roll_windows']:
        d[f'{col}_roll{w}_mean'] = d[col].rolling(window=w).mean()
    return d

def build_model_frame(df, target_col, other_col, cfg=CONFIG):
    d = add_lag_features(df, target_col, cfg)
    d[f'{target_col}_future'] = d[target_col].shift(-cfg['horizon'])
    target_hist_cols = [f'{target_col}_lag{l}' for l in cfg['lags']] + \
                        [f'{target_col}_roll{w}_mean' for w in cfg['roll_windows']] + [target_col]
    proxy = {'CO(GT)': 'PT08.S1(CO)', 'C6H6(GT)': 'PT08.S2(NMHC)'}
    other_cols = ['NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)',
                  other_col, proxy.get(other_col)]
    other_cols = [c for c in other_cols if c and c != target_col]
    groups = {'time': ['Hour_sin','Hour_cos','DoW_sin','DoW_cos'],
              'target_history': target_hist_cols, 'other_pollutants': other_cols,
              'weather': ['T','RH','AH']}
    keep = sum(groups.values(), []) + [f'{target_col}_future', 'DateTime']
    return d[keep].copy(), groups

def time_split(model_df, target_future_col, cfg=CONFIG):
    n = len(model_df)
    split_date = model_df['DateTime'].iloc[int(n * cfg['split_fraction'])]
    train_pool = model_df[model_df['DateTime'] < split_date].dropna()
    test_pool  = model_df[model_df['DateTime'] >= split_date].dropna()
    feat_cols = [c for c in model_df.columns if c not in (target_future_col, 'DateTime')]
    return (train_pool[feat_cols], test_pool[feat_cols],
            train_pool[target_future_col], test_pool[target_future_col],
            test_pool['DateTime'], split_date)

def _score(y_true, y_pred, n=None):
    return {'n': n if n is not None else len(y_true), 'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)), 'R2': r2_score(y_true, y_pred)}

## Four Baselines


1. **Persistence**  assume the value 6 hours from now equals the value right now.
2. **Same-hour-yesterday**  assume it equals the value 24 hours before the *target* time.
3. **Training mean**  assume it equals the historical average, ignoring time entirely.
4. **Hour-of-week average** assume it equals the historical average for that specific (weekday, hour) combination, computed from training data only. 

In [9]:
def compute_baselines(df, target_col, model_df, X_test, y_test, dt_test, y_train, cfg=CONFIG):
    horizon = cfg['horizon']; results = {}
    results['Persistence'] = _score(y_test.values, X_test[target_col].values)

    shift_amt = 24 - horizon
    sh_pred = df.set_index('DateTime')[target_col].shift(shift_amt).reindex(dt_test).values
    mask = ~np.isnan(sh_pred)
    results['Same-hour-yesterday'] = _score(y_test.values[mask], sh_pred[mask], n=int(mask.sum()))

    results['Training mean'] = _score(y_test.values, np.full(len(y_test), y_train.mean()))

    train_dt = model_df.loc[y_train.index, 'DateTime'] + pd.Timedelta(hours=horizon)
    test_dt = dt_test + pd.Timedelta(hours=horizon)
    hw_avg = pd.Series(y_train.values, index=pd.MultiIndex.from_arrays(
        [train_dt.dt.dayofweek, train_dt.dt.hour])).groupby(level=[0,1]).mean()
    hw_pred = np.array([hw_avg.get((d,h), y_train.mean()) for d,h in zip(test_dt.dt.dayofweek, test_dt.dt.hour)])
    results['Hour-of-week average'] = _score(y_test.values, hw_pred)
    return results

def print_baselines(name, b):
    print(f"\n{name} baselines:")
    for k,v in b.items():
        print(f"  {k:22s} n={v['n']:5d}  MAE={v['MAE']:.3f}  RMSE={v['RMSE']:.3f}  R2={v['R2']:.3f}")

## Model Selection

Four model families (Linear Regression, Ridge, Random Forest, XGBoost), each tried on the raw target and on `log1p(target)`. Random Forest and XGBoost each get a light `RandomizedSearchCV` (`TimeSeriesSplit`, 3 folds, 6 iterations) to find reasonable hyperparameters. Then **all eight variants** are compared on a common footing: a manual walk-forward `TimeSeriesSplit` (4 folds) over the training set only, with every prediction converted back to the original scale with `expm1()` before scoring  so a log1p model and a raw model are judged on the same units, not on log-space error for one and real-units error for the other.

The test set is not touched during any of this. It is used exactly once, after the best variant is selected, for the final reported numbers.

In [10]:
def tune_tree_model(model_cls, grid, X_train, y_train_t, cfg=CONFIG):
    search = RandomizedSearchCV(model_cls(random_state=cfg['random_state']), grid, n_iter=6,
                                 cv=TimeSeriesSplit(n_splits=3), scoring='neg_mean_squared_error',
                                 random_state=cfg['random_state'], n_jobs=-1)
    search.fit(X_train, y_train_t)
    return search.best_params_

def cv_compare_variants(X_train, y_train, cfg=CONFIG):
    rf_grid = {'n_estimators':[100,200], 'max_depth':[5,10,None], 'min_samples_leaf':[1,2,5]}
    xgb_grid = {'n_estimators':[100,200], 'max_depth':[3,5,7], 'learning_rate':[0.05,0.1,0.2]}
    rf_raw_p = tune_tree_model(RandomForestRegressor, rf_grid, X_train, y_train, cfg)
    rf_log_p = tune_tree_model(RandomForestRegressor, rf_grid, X_train, np.log1p(y_train), cfg)
    xgb_raw_p = tune_tree_model(XGBRegressor, xgb_grid, X_train, y_train, cfg)
    xgb_log_p = tune_tree_model(XGBRegressor, xgb_grid, X_train, np.log1p(y_train), cfg)

    variants = {
        'Linear Regression (raw)':   (LinearRegression(), False),
        'Linear Regression (log1p)': (LinearRegression(), True),
        'Ridge (raw)':                (Ridge(alpha=1.0, random_state=cfg['random_state']), False),
        'Ridge (log1p)':              (Ridge(alpha=1.0, random_state=cfg['random_state']), True),
        'Random Forest (raw)':       (RandomForestRegressor(**rf_raw_p, random_state=cfg['random_state']), False),
        'Random Forest (log1p)':     (RandomForestRegressor(**rf_log_p, random_state=cfg['random_state']), True),
        'XGBoost (raw)':             (XGBRegressor(**xgb_raw_p, random_state=cfg['random_state']), False),
        'XGBoost (log1p)':           (XGBRegressor(**xgb_log_p, random_state=cfg['random_state']), True),
    }
    tscv = TimeSeriesSplit(n_splits=cfg['cv_splits'])
    rows = []
    for name, (estimator, use_log) in variants.items():
        fold_scores = []
        for tr_idx, val_idx in tscv.split(X_train):
            Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            est = clone(estimator)
            est.fit(Xtr, np.log1p(ytr) if use_log else ytr)
            pred = est.predict(Xval)
            if use_log: pred = np.expm1(pred)
            fold_scores.append(_score(yval.values, pred))
        rows.append({'Variant': name, 'CV_MAE': np.mean([f['MAE'] for f in fold_scores]),
                     'CV_RMSE': np.mean([f['RMSE'] for f in fold_scores]),
                     'CV_R2': np.mean([f['R2'] for f in fold_scores])})
    return pd.DataFrame(rows).sort_values('CV_R2', ascending=False).reset_index(drop=True), variants

In [11]:
def grouped_permutation_importance(model, use_log, X_test, y_test, groups, n_repeats=10, seed=42):
    rng = np.random.RandomState(seed)
    base_pred = model.predict(X_test)
    if use_log: base_pred = np.expm1(base_pred)
    base_score = r2_score(y_test, base_pred)
    rows = []
    for gname, cols in groups.items():
        cols = [c for c in cols if c in X_test.columns]
        if not cols: continue
        drops = []
        for _ in range(n_repeats):
            X_shuf = X_test.copy()
            for c in cols: X_shuf[c] = rng.permutation(X_shuf[c].values)
            pred = model.predict(X_shuf)
            if use_log: pred = np.expm1(pred)
            drops.append(base_score - r2_score(y_test, pred))
        rows.append({'Group': gname, 'Mean_R2_drop': np.mean(drops), 'Std_R2_drop': np.std(drops)})
    return pd.DataFrame(rows).sort_values('Mean_R2_drop', ascending=False).reset_index(drop=True)

def ablation(model_factory, use_log, X_train, y_train, X_test, y_test, groups, full_r2):
    rows = [{'Removed_group': '(none  full model)', 'R2': full_r2, 'Delta_vs_full': 0.0}]
    for gname, cols in groups.items():
        cols = [c for c in cols if c in X_train.columns]
        keep = [c for c in X_train.columns if c not in cols]
        if not keep: continue
        est = model_factory()
        est.fit(X_train[keep], np.log1p(y_train) if use_log else y_train)
        pred = est.predict(X_test[keep])
        if use_log: pred = np.expm1(pred)
        r2 = r2_score(y_test, pred)
        rows.append({'Removed_group': gname, 'R2': r2, 'Delta_vs_full': r2 - full_r2})
    return pd.DataFrame(rows)

def spike_analysis(y_test, pred, threshold, label='Value'):
    is_spike = y_test.values > threshold
    out = {f'{label} <= {threshold} (n={int((~is_spike).sum())})': _score(y_test.values[~is_spike], pred[~is_spike]),
           f'{label} > {threshold}  (n={int(is_spike.sum())})': _score(y_test.values[is_spike], pred[is_spike])}
    return pd.DataFrame(out).T

def run_full_pipeline(df, target_col, other_col, spike_threshold, label):
    model_df, groups = build_model_frame(df, target_col, other_col)
    X_train, X_test, y_train, y_test, dt_test, split_date = time_split(model_df, f'{target_col}_future')
    print(f"=== {label} === split date: {split_date} | train: {X_train.shape} | test: {X_test.shape}")

    baselines = compute_baselines(df, target_col, model_df, X_test, y_test, dt_test, y_train)
    print_baselines(label, baselines)

    cv_table, variants = cv_compare_variants(X_train, y_train)
    print(f"\n{label} CV comparison (walk-forward, original-scale R2):")
    print(cv_table.to_string(index=False))

    best_name = cv_table.iloc[0]['Variant']
    best_template, use_log = variants[best_name]
    final_model = clone(best_template)
    final_model.fit(X_train, np.log1p(y_train) if use_log else y_train)
    test_pred = final_model.predict(X_test)
    if use_log: test_pred = np.expm1(test_pred)
    final_score = _score(y_test.values, test_pred)
    print(f"\n{label} SELECTED: {best_name}  |  FINAL TEST SCORE (touched once): {final_score}")

    imp = grouped_permutation_importance(final_model, use_log, X_test, y_test, groups)
    print(f"\n{label} grouped permutation importance:\n{imp}")

    abl = ablation(lambda: clone(best_template), use_log, X_train, y_train, X_test, y_test, groups, final_score['R2'])
    print(f"\n{label} ablation (interpretive  test set reused here for explanation, not re-selection):\n{abl}")

    spikes = spike_analysis(y_test, test_pred, spike_threshold, label=label)
    print(f"\n{label} spike analysis:\n{spikes}")

    plt.figure(figsize=(12,4))
    plt.plot(dt_test, y_test.values, label=f'Actual {label} (t+{CONFIG["horizon"]}h)', alpha=0.8)
    plt.plot(dt_test, test_pred, label='Predicted', alpha=0.8)
    plt.title(f'{label}: actual vs predicted (test period, datetime axis)'); plt.legend()
    plt.tight_layout(); plt.show()

    residuals = y_test.values - test_pred
    plt.figure(figsize=(12,3.5))
    plt.plot(dt_test, residuals, color='indianred'); plt.axhline(0, color='black', linewidth=0.8)
    plt.title(f'{label}: residuals over time (test period)'); plt.tight_layout(); plt.show()

    return dict(baselines=baselines, cv_table=cv_table, best_name=best_name, final_score=final_score,
                imp=imp, abl=abl, spikes=spikes, model=final_model, use_log=use_log, X_test=X_test)

## Results CO(GT)

In [12]:
co_results = run_full_pipeline(df, 'CO(GT)', 'C6H6(GT)', CONFIG['spike_threshold_co'], 'CO(GT)')

**Interpretation.** The Hour-of-week average baseline alone reaches R2 = 0.40  since it already encodes the daily/weekly rhythm found in the EDA. The selected model (Random Forest, log1p target) reaches R2 = 0.46 on the untouched test set, beating every baseline.
Persistence (R2 = -0.90) and the training mean (R2 = -002) both fail outright at a 6-hour horizon 

**Grouped permutation importance** shows `time` (the cyclical hour/weekday features) causing by far the largest score drop when shuffled, well ahead of the target's own history; `other_pollutants` and `weather` show a small *negative* drop, meaning shuffling them very slightly helps.

**The ablation study confirms this isn't noise**: removing `other_pollutants` or `weather` from the feature set *increases* test R2 above the full model's score, while removing `time` or `target_history` sharply reduces it. In other words, the leanest model  cyclical time features plus the target's own lag/rolling history  generalizes better than the full feature set here; the extra sensor columns appear to add noise rather than signal for this specific forecast. (This ablation reuses the test set for explanation after the model was already selected via CV  it is not used to pick a different final model, which would reopen the test set to selection.)

**Spike analysis** (CO > 4, the top ~9% of observed test values, n=137): MAE roughly 2.5x higher than the non-spike bucket (1.55 vs 0.62), and R2 goes sharply negative on the spike subset alone a precise, quantified version of "the model under-predicts rare pollution spikes," not just a visual impression from the plot.

## Results  C6H6(GT)

In [13]:
c6h6_results = run_full_pipeline(df, 'C6H6(GT)', 'CO(GT)', CONFIG['spike_threshold_c6h6'], 'C6H6(GT)')

**Interpretation.** The same pattern repeats: `time` dominates grouped permutation importance, and removing `other_pollutants`/`weather` again improves test R2 in the ablation  this is not specific to one pollutant, it's a property of this feature set and forecast horizon. 

## Results  CO(GT) vs C6H6(GT)


In [14]:
comparison = pd.DataFrame([
    {'Target': 'CO(GT)',   'Best model': co_results['best_name'],
     'Hour-of-week baseline R2': co_results['baselines']['Hour-of-week average']['R2'],
     'Final test R2': co_results['final_score']['R2'], 'Final test MAE': co_results['final_score']['MAE']},
    {'Target': 'C6H6(GT)', 'Best model': c6h6_results['best_name'],
     'Hour-of-week baseline R2': c6h6_results['baselines']['Hour-of-week average']['R2'],
     'Final test R2': c6h6_results['final_score']['R2'], 'Final test MAE': c6h6_results['final_score']['MAE']},
])
print(comparison.to_string(index=False))

Observation :
**CO(GT) forecasts better than C6H6(GT) (R2 0.46 vs 0.38), even though `C6H6(GT)` has a 0.98-correlated proxy sensor (`PT08.S2`) and `CO(GT)` does not.** That proxy correlation is a same-instant relationship; it does not transfer to a 6-hour-ahead forecast. Strong correlation is not the same thing as forecasting power, and that conclusion is now backed by a CV-selected model and a fair, identical pipeline on both sides rather than a single train/test split.

## Why Leakage Matters


In [15]:
leak_cols = ['CO(GT)','PT08.S1(CO)','C6H6(GT)','PT08.S2(NMHC)','NOx(GT)','PT08.S3(NOx)',
             'NO2(GT)','PT08.S4(NO2)','PT08.S5(O3)','T','RH','AH']
d_leak = df[leak_cols].dropna()
X_leak, y_leak = d_leak.drop(columns=['CO(GT)']), d_leak['CO(GT)']
Xtr, Xte, ytr, yte = train_test_split(X_leak, y_leak, test_size=0.2, random_state=RNG, shuffle=True)
rf_leak = RandomForestRegressor(n_estimators=150, random_state=RNG).fit(Xtr, ytr)
pred_leak = rf_leak.predict(Xte)
print(f"Leaky setup (same-timestamp features, shuffled split): "
      f"R2={r2_score(yte,pred_leak):.3f}  MAE={mean_absolute_error(yte,pred_leak):.3f}  "
      f"RMSE={np.sqrt(mean_squared_error(yte,pred_leak)):.3f}  (n={len(d_leak)})")
print(f"Honest 6h-ahead forecast (this notebook's actual result): R2={co_results['final_score']['R2']:.3f}")

**The gap is the leakage, quantified on the exact same cleaned data:** R2 = 0.93 for the leaky same-timestamp/shuffled-split setup, versus R2 = 0.46 for the honest 6-hour-ahead forecast above. Both runs use identical cleaning  the only difference is the task definition and the split  so this 0.47-point gap isolates the effect of the leakage itself, not a side effect of also changing how missing data was handled.

**Why it happens:** a shuffled split lets the model train on rows sitting hours from a test row; because adjacent hours are strongly autocorrelated, the model is effectively tested on near-duplicates of data it already saw. Predicting a same-timestamp reading from other simultaneous sensor readings is also a fundamentally easier problem than forecasting forward in time  it's closer to sensor calibration than to forecasting.

**Prevention, demonstrated in this notebook, not just stated:**
1. Split strictly by time (`time_split` above cuts on a real timestamp, never shuffles).
2. Always shift the target forward relative to the features (`{target}_future = {target}.shift(-horizon)`), never share a timestamp.
3. Fit only on the training fold before scoring anything `cv_compare_variants` and the final model both fit on `X_train`/`y_train` exclusively; the test set is touched exactly once, for the final numbers above.

## Key Findings

- By limiting forward-filling to gaps of three hours or less, the data cleaning process avoided fabricating approximately 1,500 hours of CO(GT) data.
- Exploratory data analysis revealed that CO(GT) peaks on Fridays, with an average of 2.53 mg/m³, and reaches a weekly low on Sundays at 1.43 mg/m³. Daily weekday peaks align with expected traffic hours around 8–9 AM and 6–8 PM.
- The selected model for CO(GT) was a Random Forest using a `log1p` target, which achieved an R² of 0.46 on the test set and outperformed the strongest baseline (Hour-of-week average, R² = 0.40).
- CO(GT) proved easier to forecast than C6H6(GT), scoring an R² of 0.46 versus 0.38. This demonstrated that the 0.98 simultaneous correlation between C6H6(GT) and its proxy sensor does not translate to strong 6-hour forecasting power.
- The ablation study indicated that excluding other pollutants and weather data actually improved the test R². This suggests the leanest combination of cyclical time features and target history generalizes better than the full feature set.
- The model severely under-predicts rare, high-pollution spikes where CO > 4, resulting in an R² that turns sharply negative for that specific subset.
- A controlled test isolated the impact of data leakage, showing that a flawed setup with same-timestamp features and a shuffled split artificially inflated the R² by 0.47 points compared to the honest chronological forecast.

## Limitations

- **Data Loss:** Because long gaps are left as `NaN`, roughly 15% of the CO(GT) hours are entirely missing and dropped from the modeling pool. These drops are highly concentrated in April, July, August, and October.
- **Missing Covariates:** The forecast relies purely on the sensors' past and current readings. It lacks access to external predictors such as forward-looking weather forecasts or traffic data, which caps its predictive ceiling.
- **Single Year Dependency:** With only one year of data, the model risks memorizing unique seasonal quirks. This limitation forced the deliberate exclusion of month and season features to prevent overfitting.

## Improvements for Future Work

- Develop a dedicated anomaly or spike detection model designed specifically to catch the high-pollution events that the base regression model struggles to predict.
- Integrate real external forecast data, such as weather predictions for the target time window, to provide signal beyond what the sensors' self-history can offer.